In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "LogReg":  (LogisticRegression(), {"C": [0.01, 0.1, 1.0]}, 1),
    "XGBoost": (XGBClassifier(eval_metric="logloss", verbosity=0), 
                {"n_estimators": [50, 100], "max_depth": [3, 5]}, -1),
}



import pandas as pd
import numpy as np

np.random.seed(42)

# 500 Business Days
dates = pd.date_range("2023-01-01", periods=500, freq="B")
n = len(dates)

# Realistische Returns
returns = np.random.normal(0.0003, 0.012, n)
for i in range(1, n):
    returns[i] += 0.05 * returns[i-1]

# Features
lag1 = np.roll(returns, 1)
lag2 = np.roll(returns, 2)
vol5 = pd.Series(returns).rolling(5).std().values

# Target: 1 = positiver Return, 0 = negativer Return
direction = (returns > 0).astype(int)    # → 0, 1

df = pd.DataFrame({
    "Date": dates,
    "lag1": lag1,
    "lag2": lag2,
    "vol5": vol5,
    "direction": direction
})

df = df.iloc[5:].reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Zeitraum: {df['Date'].iloc[0].date()} bis {df['Date'].iloc[-1].date()}")
print(f"Klassen: {df['direction'].value_counts().to_dict()}")





import sys
# sys.path not needed after pip install qtml
from qtml.Time_Series_Univariate import rolling_forecast


predictions = rolling_forecast(
    df, 
    target="direction",
    models=models,
    test_start="2023-06-01",
    test_end="2024-12-01"
)